# 01 Chris - A few experiments

This notebook tries a first logistic regressic, after a basic grid search for lambda and gamma values, involving a cross-validation on a undersampled subdataset.

In [21]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'implementations.py').exists():
    PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from implementations import reg_logistic_regression

In [27]:
DATA_PATH = PROJECT_ROOT / 'dataset/prepared/selected_random_over_dataset.npz'
data = np.load(DATA_PATH, allow_pickle=True)

# Reuse notebook 01's processed features and original class distribution.
x_train_clean = data['x_train_raw']
y_train_clean = data['y_train_raw']
x_test_clean = data['x_test']
test_ids = data['test_ids']
if set(np.unique(y_train_clean)) == {-1, 1}:
    y_train_binary = (y_train_clean == 1).astype(int)
else:
    y_train_binary = y_train_clean.astype(int)

print(f"Shape X train: {x_train_clean.shape}")
print(f"Shape Y train: {y_train_binary.shape}")
print(f"Class counts: {np.bincount(y_train_binary)}")
print(f"Shape X test: {x_test_clean.shape}")

Shape X train: (328135, 140)
Shape Y train: (328135,)
Class counts: [299160  28975]
Shape X test: (109379, 140)


In [28]:
# Adding bias column (1)
tx_train = np.c_[np.ones((x_train_clean.shape[0], 1)), x_train_clean]
tx_test = np.c_[np.ones((x_test_clean.shape[0], 1)), x_test_clean]

print(f"Shape tX train (with biais): {tx_train.shape}")

Shape tX train (with biais): (328135, 141)


> Here are some functions
- F1 (based on precision and recall), metric to evaluate the result;
- An undersampling algorithm to cope with the fact that we only have 8.8% of positive cases in the original dataset;
- A cross-validation algorithm (as in Labs sessions);
- A grid search based on the undersampled dataset and cross-validation algorithm.

In [34]:
def compute_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(y_true)

    return accuracy, precision, recall, f1


def undersample(x, y, seed=1):
    rng = np.random.default_rng(seed)
    idx_1 = np.where(y == 1)[0]
    idx_0 = np.where(y == 0)[0]

    min_size = min(len(idx_1), len(idx_0))

    idx_1_sub = rng.choice(idx_1, min_size, replace=False)
    idx_0_sub = rng.choice(idx_0, min_size, replace=False)

    idx = np.concatenate([idx_1_sub, idx_0_sub])
    rng.shuffle(idx)

    return x[idx], y[idx]


def stratified_split(x, y, validation_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_indices = []
    validation_indices = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        rng.shuffle(indices)
        validation_size = int(len(indices) * validation_fraction)
        validation_indices.extend(indices[:validation_size])
        train_indices.extend(indices[validation_size:])
    rng.shuffle(train_indices)
    rng.shuffle(validation_indices)
    return x[train_indices], y[train_indices], x[validation_indices], y[validation_indices]


def best_f1_threshold(y_true, probabilities):
    thresholds = np.unique(probabilities)
    f1_scores = [
        compute_metrics(y_true, probabilities >= threshold)[3]
        for threshold in thresholds
    ]
    return float(thresholds[np.argmax(f1_scores)])


def build_poly(x, degree):
    poly = np.ones((len(x), 1))
    for j in range(1, degree + 1):
        poly = np.c_[poly, np.power(x, j)]
    return poly


def build_k_indices(y, k_fold, seed=1):
    num_row = y.shape[0]
    interval = int(num_row / k_fold)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(num_row)
    k_indices = [indices[k * interval : (k + 1) * interval] for k in range(k_fold)]
    return np.array(k_indices)


def cross_validation(y, x, k_indices, k, lambda_, gamma, max_iters, model_func):
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)]
    tr_indice = tr_indice.reshape(-1)

    x_tr, y_tr = x[tr_indice], y[tr_indice]
    x_te, y_te = x[te_indice], y[te_indice]

    # Keep threshold tuning on the original class distribution.
    x_fit, y_fit, x_threshold, y_threshold = stratified_split(
        x_tr, y_tr, validation_fraction=0.2, seed=42 + k
    )
    x_fit, y_fit = undersample(x_fit, y_fit, seed=42 + k)

    initial_w = np.zeros(x_fit.shape[1])
    w, _ = model_func(y_fit, x_fit, lambda_, initial_w, max_iters, gamma)

    threshold_probs = 1 / (1 + np.exp(-x_threshold.dot(w)))
    threshold = best_f1_threshold(y_threshold, threshold_probs)
    y_pred_probs = 1 / (1 + np.exp(-x_te.dot(w)))
    y_pred = (y_pred_probs >= threshold).astype(int)

    return compute_metrics(y_te, y_pred)


def grid_search(y, x, k_fold, lambdas, gammas, max_iters, model_func):
    k_indices = build_k_indices(y, k_fold)
    best_f1 = -1.0
    best_params = None

    for lambda_ in lambdas:
        for gamma in gammas:
            f1_scores = []
            for k in range(k_fold):
                _, _, _, f1 = cross_validation(
                    y, x, k_indices, k, lambda_, gamma, max_iters, model_func
                )
                f1_scores.append(f1)

            avg_f1 = np.mean(f1_scores)

            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_params = (lambda_, gamma)

    return best_params, best_f1

> To avoid to much computing, we select a few lambdas and gammas values.

In [36]:
# Compact grid search using threshold-tuned cross-validation.
lambdas = [1e-4, 1e-3, 1e-2]
gammas = [0.05, 0.1]
max_iters = 200
k_fold = 3

best_params, best_f1 = grid_search(
    y=y_train_binary,
    x=tx_train,
    k_fold=k_fold,
    lambdas=lambdas,
    gammas=gammas,
    max_iters=max_iters,
    model_func=reg_logistic_regression,
)

best_lambda, best_gamma = best_params
print(f"Best parameters: lambda = {best_lambda}, gamma = {best_gamma}")
print(f"Average tuned F1-score: {best_f1:.4f}")

Best parameters: lambda = 0.0001, gamma = 0.1
Average tuned F1-score: 0.3935


In [31]:
# Reserve a real-distribution validation split for threshold selection.
x_fit, y_fit, x_threshold, y_threshold = stratified_split(
    tx_train, y_train_binary, validation_fraction=0.2, seed=42
)
x_fit_sub, y_fit_sub = undersample(x_fit, y_fit, seed=42)

initial_w = np.zeros(x_fit_sub.shape[1])
final_max_iters = 2000

w_threshold, _ = reg_logistic_regression(
    y_fit_sub,
    x_fit_sub,
    best_lambda,
    initial_w,
    final_max_iters,
    best_gamma,
)
threshold_probs = 1 / (1 + np.exp(-x_threshold.dot(w_threshold)))
tuned_threshold = best_f1_threshold(y_threshold, threshold_probs)
print(f"Tuned validation threshold: {tuned_threshold:.4f}")

# Retrain on an undersampled full training set after threshold selection.
x_tr_final, y_tr_final = undersample(tx_train, y_train_binary, seed=42)
initial_w = np.zeros(x_tr_final.shape[1])
w_final, final_loss = reg_logistic_regression(
    y_tr_final,
    x_tr_final,
    best_lambda,
    initial_w,
    final_max_iters,
    best_gamma,
)

print(f"Final loss: {final_loss:.4f}")

Tuned validation threshold: 0.7100
Final loss: 0.4710


> The classification threshold is tuned on a held-out validation split with the original class distribution. Undersampling is applied only to the training portion.

In [32]:
# Predict on the test set using the validation-tuned threshold.
y_test_probs = 1 / (1 + np.exp(-tx_test.dot(w_final)))
y_test_preds_binary = (y_test_probs >= tuned_threshold).astype(int)

# Reconvert to (-1, 1) for AIcrowd.
y_test_preds_aicrowd = np.where(y_test_preds_binary == 0, -1, 1)

In [33]:
from helpers import create_csv_submission

submission_path = PROJECT_ROOT / 'submission.csv'
create_csv_submission(test_ids, y_test_preds_aicrowd, submission_path)

print(f"Submission file written to: {submission_path}")

Submission file written to: /Users/mohamedzirar/Documents/year4/ML/ML_project1/submission.csv
